# 01.3 · Hub-and-Spoke 模式（asyncio 并行）

> Orchestrator 并行调度多个 Agent，再合成结果。


In [1]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


---
## Part 3 · Hub-and-Spoke 模式（asyncio 并行）

Orchestrator 用 `asyncio.gather` **并行**调用多个 Agent，再合成结果。

```
用户
 │
 ▼
Orchestrator
 ├─── asyncio.gather ──▶ RetrieverAgent  (0.3s)
 │                  ──▶ MetadataAgent   (0.1s)  ← 同时运行！
 │                  ──▶ SummaryAgent    (0.2s)
 │         ▼ 全部完成后
 └─── GeneratorAgent（拿到所有结果再生成）
```

In [3]:
# ── 日志辅助 ─────────────────────────────────────────────────────────────────
_t0 = time.perf_counter()

def log(stage: str, msg: str):
    print(f"  {time.perf_counter() - _t0:6.3f}s │ [{stage:>12}] {msg}", flush=True)

# ── 异步 Agent 定义 ──────────────────────────────────────────────────────────
async def async_retrieve(query: str) -> list[dict]:
    log("Retriever", f"▶ 开始 │ query={query[:18]!r}")
    t0 = time.perf_counter()
    await asyncio.sleep(0.3)
    hits = fake_retrieve(query, top_k=5, latency=0)
    log("Retriever", f"✓ 完成 {time.perf_counter()-t0:.2f}s │ query={query[:18]!r} │ "
        f"hits={len(hits)} top_score={hits[0]['score']}")
    return hits

async def async_metadata(query: str) -> dict:
    log("Metadata", f"▶ 开始 │ query={query[:18]!r}")
    t0 = time.perf_counter()
    await asyncio.sleep(0.1)
    meta = {"doc_count": random.randint(100, 500), "last_updated": "2025-01"}
    log("Metadata", f"✓ 完成 {time.perf_counter()-t0:.2f}s │ query={query[:18]!r} │ "
        f"doc_count={meta['doc_count']} last_updated={meta['last_updated']}")
    return meta

async def async_summary_hint(query: str) -> str:
    log("SummaryHint", f"▶ 开始 │ query={query[:18]!r}")
    t0 = time.perf_counter()
    await asyncio.sleep(0.2)
    hint = f"请用简洁语言回答：{query}"
    log("SummaryHint", f"✓ 完成 {time.perf_counter()-t0:.2f}s │ query={query[:18]!r} │ hint={hint[:28]!r}")
    return hint

async def orchestrator(query: str) -> dict:
    global _t0
    _t0 = time.perf_counter()

    print("📤 请求阶段")
    log("Orchestrator", f"收到请求 │ query={query[:18]!r}")
    print("\n📊 并行阶段（asyncio.gather — 3 个 Agent 同时运行）")
    print("  " + "─" * 72)
    log("Orchestrator", "并行调用 Retriever + Metadata + SummaryHint ...")

    t_parallel = time.perf_counter()
    hits, meta, hint = await asyncio.gather(
        async_retrieve(query),
        async_metadata(query),
        async_summary_hint(query),
    )
    parallel_elapsed = round(time.perf_counter() - t_parallel, 3)

    serial_parallel = 0.3 + 0.1 + 0.2          # 若串行执行三者的耗时
    parallel_savings = round(serial_parallel - max(0.3, 0.1, 0.2), 3)

    log("Orchestrator",
        f"并行阶段完成 {parallel_elapsed}s │ hits={len(hits)} doc_count={meta['doc_count']} "
        f"hint={hint[:25]!r}...")
    print("  " + "─" * 72)

    print("\n📊 串行阶段（依赖并行结果）")
    print("  " + "─" * 72)
    log("Orchestrator", "开始生成（需 hits + hint）...")
    t_gen = time.perf_counter()
    await asyncio.sleep(0.5)
    answer = fake_generate(hint, hits, latency=0)
    gen_elapsed = round(time.perf_counter() - t_gen, 3)
    log("Generator", f"✓ 完成 {gen_elapsed}s │ query={query[:18]!r} │ answer={answer[:40]}...")
    print("  " + "─" * 72)

    elapsed = round(time.perf_counter() - _t0, 3)
    serial_total = round(serial_parallel + 0.5, 3)

    return {
        "answer": answer,
        "meta": meta,
        "latency_s": elapsed,
        "parallel_elapsed_s": parallel_elapsed,
        "gen_elapsed_s": gen_elapsed,
        "parallel_savings_s": parallel_savings,
        "serial_total_s": serial_total,
    }

# ── 运行 ─────────────────────────────────────────────────────────────────────
query = "什么是向量数据库？"
result = asyncio.run(orchestrator(query))

print("\n📈 汇总")
print(f"  总延迟：{result['latency_s']}s（并行 {result['parallel_elapsed_s']}s + 生成 {result['gen_elapsed_s']}s）")
print(f"  并行节省：{result['parallel_savings_s']}s（并行阶段串行需 {0.3+0.1+0.2}s，实际 max={max(0.3,0.1,0.2)}s）")
print(f"  若全串行：{result['serial_total_s']}s → 本次快 {round(result['serial_total_s'] - result['latency_s'], 3)}s")
print(f"  文档数：{result['meta']['doc_count']}")
print(f"  答案：{result['answer'][:60]}...")

📤 请求阶段
   0.000s │ [Orchestrator] 收到请求 │ query='什么是向量数据库？'

📊 并行阶段（asyncio.gather — 3 个 Agent 同时运行）
  ────────────────────────────────────────────────────────────────────────
   0.001s │ [Orchestrator] 并行调用 Retriever + Metadata + SummaryHint ...
   0.002s │ [   Retriever] ▶ 开始 │ query='什么是向量数据库？'
   0.002s │ [    Metadata] ▶ 开始 │ query='什么是向量数据库？'
   0.003s │ [ SummaryHint] ▶ 开始 │ query='什么是向量数据库？'
   0.112s │ [    Metadata] ✓ 完成 0.11s │ query='什么是向量数据库？' │ doc_count=216 last_updated=2025-01
   0.205s │ [ SummaryHint] ✓ 完成 0.20s │ query='什么是向量数据库？' │ hint='请用简洁语言回答：什么是向量数据库？'
   0.315s │ [   Retriever] ✓ 完成 0.31s │ query='什么是向量数据库？' │ hits=5 top_score=0.998
   0.316s │ [Orchestrator] 并行阶段完成 0.315s │ hits=5 doc_count=216 hint='请用简洁语言回答：什么是向量数据库？'...
  ────────────────────────────────────────────────────────────────────────

📊 串行阶段（依赖并行结果）
  ────────────────────────────────────────────────────────────────────────
   0.318s │ [Orchestrator] 开始生成（需 hits + hint）...
   0.829s │ [   Generator

> **关键观察**：
> - 3 个 Agent 并行，总时间 ≈ `max(0.3, 0.1, 0.2) + 0.5 = 0.8s`
> - 串行则需 `0.3 + 0.1 + 0.2 + 0.5 = 1.1s`
> - `asyncio.gather` 是实现并行的关键

## 📖 讲义

# Part 3
## 通信机制与状态管理

---

# 四种通信机制

| 机制 | 代表技术 | 延迟 | 耦合度 | 适用场景 |
|------|----------|------|--------|----------|
| **Sync RPC** | HTTP / gRPC | 低 | 高 | 实时决策、简单编排 |
| **Async Queue** | Redis / Kafka / SQS | 中 | 低 | 耗时操作、高并发 |
| **Pub-Sub** | SNS / Kafka Topic | 中 | 很低 | 广播事件、多订阅方 |
| **Shared Storage** | S3 + DB + 信号 | 高 | 很低 | 大文件、artifacts 共享 |

---

# 选择通信机制的决策树

```
需要立即拿到结果？
 │
 ├─ YES → 延迟是否 < 100ms？
 │          ├─ YES → gRPC / HTTP
 │          └─ NO  → HTTP + polling / WebSocket
 │
 └─ NO → 任务是否耗时 > 1s？
           ├─ YES → Message Queue（Redis / Kafka）
           └─ NO  → 可以用 async HTTP + callback
```

**黄金法则**
- 用户直接等待 → 同步 RPC
- 后台批处理 → 异步队列
- 广播通知 → Pub-Sub
- 大文件传递 → Shared Storage

---

# 状态、记忆与幂等

<div class="columns">
<div>

### Agent State 分类

**短期状态（任务内）**
- 随消息一起传递
- 无需持久化
- 示例：当前检索结果

**长期状态（跨会话）**
- 存 DB / Redis
- 需要 TTL 和生命周期管理
- 示例：用户偏好、对话历史

</div>
<div>

### 幂等设计（关键！）

```python
# 每条消息带唯一 ID
job = {
    "request_id": "uuid-xxx",
    "query": "...",
}

# 处理前检查是否已处理
if redis.get(f"done:{job['request_id']}"):
    return  # 跳过，防止重复

process(job)
redis.setex(f"done:{job['request_id']}", 3600, 1)
```

**为什么必须幂等？**
网络重试 + 消费者崩溃恢复都会重复投递消息

</div>
</div>

---

# Backpressure 与流量控制

```
生产者速率 >> 消费者速率 → 队列积压 → OOM / 延迟飙升
```

**应对策略**

| 策略 | 实现 | 效果 |
|------|------|------|
| 队列长度限制 | `maxmemory-policy allkeys-lru` | 防止内存耗尽 |
| 消费者并发控制 | Worker 数量 × 每个 Worker 并发数 | 精细控制吞吐 |
| 速率限制 | Token bucket / Redis rate limiter | 保护下游 |
| 背压信号 | 队列长度 > 阈值时拒绝新请求 | 快速失败优于慢挂 |
| Autoscaling | HPA 基于队列长度扩缩 Worker | 弹性应对流量波动 |

---

---
**导航**：[← 01.2 · Pipeline 模式](./01.2_pipeline_mode.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.4 · Blackboard 模式 →](./01.4_blackboard_mode.ipynb)
